# BibleMusically — video server (ComfyUI + open video models)

A **separate** ComfyUI from the image server, on purpose. Kaggle allows two concurrent GPU batch
sessions, so stills and clips can render at the same time on separate quota — and a video model plus
SDXL/FLUX does not fit one 16 GB card anyway.

**Enable the GPU accelerator before running** (Settings → Accelerator → GPU T4 x2). The app's
"Start & connect" already pushes with the GPU on; a GPU-less run is a cheap source update and exits
without serving.

## Which models get downloaded

Set `PACK` below. Downloading the whole catalogue is ~40 GB and would spend most of a session before
anything renders, so each package fetches only what its presets need:

| pack | models | download | runs on |
|---|---|---|---|
| `cheap_draft` | LTX-Video 2B, SD 1.5 + AnimateDiff | ~16 GB | free T4 |
| `shorts` | Wan 2.1 1.3B, LTX 2B, SD 1.5 + AnimateDiff | ~26 GB | free T4 |
| `music_video` | Wan 2.1 1.3B, LTX 2B, Wan 2.2 TI2V-5B | ~25 GB | free T4 |
| `studio_24gb` | Wan 2.1 14B, LTX 13B | ~42 GB | 24 GB Ada+ |

`studio_24gb` will **not** load on a free Kaggle T4 — Turing cannot compute in fp8, so those
checkpoints are dequantised to fp16 on load and the run dies mid-sampler. It is here for anyone
pointing this notebook at rented hardware.


## 1. ComfyUI

In [ ]:
import os, subprocess, sys

ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
COMFY = os.path.join(ROOT, 'ComfyUI')
# NOTE: enable Internet in Session options first, or these clones fail with
# "Could not resolve host: github.com".
if not os.path.isdir(COMFY):
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git', COMFY], check=True)

mgr = os.path.join(COMFY, 'custom_nodes', 'ComfyUI-Manager')
if not os.path.isdir(mgr):
    subprocess.run(['git','clone','--depth','1','https://github.com/ltdrdata/ComfyUI-Manager.git', mgr], check=False)

subprocess.run([sys.executable,'-m','pip','install','-q','-r', os.path.join(COMFY,'requirements.txt')], check=True)

ok = os.path.isfile(os.path.join(COMFY,'main.py'))
print(f'  {"OK" if ok else "FAIL"} ComfyUI at {COMFY}')
try:
    import torch
    print(f'  OK torch {torch.__version__} (cuda={torch.cuda.is_available()})')
    if torch.cuda.is_available():
        _p = torch.cuda.get_device_properties(0)
        print(f'  GPU: {_p.name}, {_p.total_memory/1e9:.0f} GB, compute {_p.major}.{_p.minor}')
        # Turing is 7.5. Everything the free tier can run is chosen around this being true, so say it
        # out loud rather than letting a later out-of-memory look like a bug.
        if _p.major < 8:
            print('  NOTE: pre-Ampere GPU — no bf16 and no fp8 arithmetic. fp8 weights load but are')
            print('        dequantised to fp16 to compute, so they save disk and VRAM, not time.')
except Exception as ex:
    ok = False
    print(f'  FAIL torch: {ex}')
print('\nComfyUI ready.' if ok else '\nSetup incomplete — see above.')


## 2. Custom nodes

In [ ]:
# Custom nodes. The three text-to-video graphs the app ships use ComfyUI's BUILT-IN nodes only
# (LTXV* and Wan reuse EmptyHunyuanLatentVideo), which is deliberate — a native graph cannot be broken
# by a custom-node version bump. Only the AnimateDiff loop graph needs a pack.
NODES = {
    'ComfyUI-AnimateDiff-Evolved': 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git',
    'ComfyUI-VideoHelperSuite':    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-GGUF':                'https://github.com/city96/ComfyUI-GGUF.git',
}
cn = os.path.join(COMFY,'custom_nodes')
for name, url in NODES.items():
    dst = os.path.join(cn, name)
    if not os.path.isdir(dst):
        subprocess.run(['git','clone','--depth','1',url,dst], check=False)
    req = os.path.join(dst,'requirements.txt')
    if os.path.isfile(req):
        subprocess.run([sys.executable,'-m','pip','install','-q','-r',req], check=False)
print('custom nodes:', list(NODES))


## 3. Choose the model package

In [ ]:
# ── Which package to download ──────────────────────────────────────────────────
# Mirrors video_models.rs in the app. Keep the two in step: the app picks a preset, and the graph it
# submits names a checkpoint that has to be on disk here.
PACK = 'music_video'   # 'cheap_draft' | 'shorts' | 'music_video' | 'studio_24gb'
API_KEY = ''           # match the app's field; blank = open server

WEIGHTS = {
    'ltx2b': [
        ('Lightricks/LTX-Video', 'ltxv-2b-0.9.8-distilled-fp8.safetensors', 'checkpoints'),
        ('comfyanonymous/flux_text_encoders', 't5xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders'),
    ],
    'wan21_t2v_1_3b': [
        # fp16 and not bf16: Turing upcasts bf16 on load, so the identically sized bf16 file costs
        # the memory the format was meant to save.
        ('Comfy-Org/Wan_2.1_ComfyUI_repackaged',
         'split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors', 'diffusion_models'),
        ('Comfy-Org/Wan_2.1_ComfyUI_repackaged',
         'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders'),
        ('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/vae/wan_2.1_vae.safetensors', 'vae'),
    ],
    'wan22_ti2v_5b': [
        ('QuantStack/Wan2.2-TI2V-5B-GGUF', 'Wan2.2-TI2V-5B-Q5_K_M.gguf', 'unet'),
        ('Comfy-Org/Wan_2.1_ComfyUI_repackaged',
         'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders'),
        ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/vae/wan2.2_vae.safetensors', 'vae'),
    ],
    'animatediff_sd15': [
        ('guoyww/animatediff', 'v3_sd15_mm.ckpt', 'animatediff_models'),
        ('runwayml/stable-diffusion-v1-5', 'v1-5-pruned-emaonly.safetensors', 'checkpoints'),
    ],
    'wan21_t2v_14b': [
        ('Comfy-Org/Wan_2.1_ComfyUI_repackaged',
         'split_files/diffusion_models/wan2.1_t2v_14B_fp8_scaled.safetensors', 'diffusion_models'),
        ('Comfy-Org/Wan_2.1_ComfyUI_repackaged',
         'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders'),
        ('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/vae/wan_2.1_vae.safetensors', 'vae'),
    ],
    'ltx13b': [
        ('Lightricks/LTX-Video', 'ltxv-13b-0.9.8-distilled-fp8.safetensors', 'checkpoints'),
        ('comfyanonymous/flux_text_encoders', 't5xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders'),
    ],
}

PACKS = {
    'cheap_draft': ['ltx2b', 'animatediff_sd15'],
    'shorts':      ['wan21_t2v_1_3b', 'ltx2b', 'animatediff_sd15'],
    'music_video': ['wan21_t2v_1_3b', 'ltx2b', 'wan22_ti2v_5b'],
    'studio_24gb': ['wan21_t2v_14b', 'ltx13b'],
}
assert PACK in PACKS, f'unknown pack {PACK}; pick one of {list(PACKS)}'
print(f'pack={PACK} -> models {PACKS[PACK]}')


## 4. Download the package's models

In [ ]:
import os
from huggingface_hub import hf_hub_download

def M(subdir, name):
    d = os.path.join(COMFY, 'models', subdir); os.makedirs(d, exist_ok=True)
    return os.path.join(d, os.path.basename(name))

_ok, _fail = [], []
def grab(repo, filename, subdir):
    dest = M(subdir, filename); label = os.path.basename(filename)
    try:
        if os.path.exists(dest) and os.path.getsize(os.path.realpath(dest)) > 1_000_000:
            print('  have', label); _ok.append(label); return
        src = hf_hub_download(repo_id=repo, filename=filename)
        if not os.path.exists(dest):
            os.symlink(src, dest)
        # A broken or LFS-pointer download is a few hundred bytes, not a model. Catching that here
        # turns a baffling load error later into an obvious download failure now.
        if os.path.getsize(os.path.realpath(dest)) < 1_000_000:
            raise RuntimeError('file suspiciously small (LFS pointer / partial download?)')
        print('  got', label); _ok.append(label)
    except Exception as ex:
        print('  FAIL', label, '->', ex); _fail.append((label, str(ex)))

seen = set()
for model_id in PACKS[PACK]:
    print(f'[{model_id}]')
    for repo, filename, subdir in WEIGHTS[model_id]:
        if (repo, filename) in seen: continue
        seen.add((repo, filename))
        grab(repo, filename, subdir)

print()
if _fail:
    # Loud and non-fatal: a missing checkpoint means the graphs that use it will fail at submit with
    # an unhelpful message, so name it now while the reason is still on screen.
    print('=' * 70)
    print(f'  {len(_fail)} FILE(S) DID NOT DOWNLOAD — presets using them will fail:')
    for label, why in _fail:
        print(f'    - {label}: {why[:120]}')
    print('=' * 70)
else:
    print(f'All {len(_ok)} model files present for pack "{PACK}".')


## 5. Serve + public tunnel

Starts ComfyUI on 8188 and opens a public tunnel. Keep this cell running.

In [ ]:
import subprocess, sys, os, re, time, threading, urllib.request

PORT = 8188
env = {**os.environ}
if API_KEY:
    env['COMFY_API_KEY'] = API_KEY  # informational; core ComfyUI has no built-in auth

# ── Batch-run guard (FIRST — before launching ComfyUI) ─────────────
# A GPU-off batch run is a cheap source-update push. ComfyUI's main.py aborts with
# 'Torch not compiled with CUDA enabled' if started without a GPU, so we must decide
# BEFORE launching it: GPU-off batch -> skip everything; GPU batch (the app's Start
# server) -> launch ComfyUI and serve.
_is_batch = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive').lower() == 'batch'

# Two independent questions, asked separately because they fail apart — and which one failed IS
# the diagnosis:
#   nvidia-smi  — does this container have a GPU and a working driver at all?
#   torch.cuda  — can the framework that runs the model actually reach it?
# A GPU-off source-update push answers no to both, and so does Kaggle declining an accelerator.
# An install step that replaced Kaggle's CUDA torch with a CPU-only wheel answers YES to the
# first and no to the second. The old guard asked only nvidia-smi and reported every "no" as an
# exhausted weekly quota — which sent people to a quota page that had 29.8 of 30 hours left on it.
_smi_rc, _smi_note = None, ''
try:
    _smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=120)
    _smi_rc = _smi.returncode
    _smi_note = (_smi.stderr or '').strip().replace('\n', ' ')[:200]
except FileNotFoundError:
    _smi_note = 'nvidia-smi is not installed on this container'
except Exception as _ex:
    _smi_note = '{}: {}'.format(type(_ex).__name__, _ex)
try:
    import torch as _t
    _torch_cuda, _torch_ver = bool(_t.cuda.is_available()), _t.__version__
except Exception as _ex:
    _torch_cuda, _torch_ver = False, 'unavailable ({})'.format(type(_ex).__name__)

# Serving is gated on torch rather than on the driver, because the model is loaded onto whatever
# device torch reports. A container that has a GPU torch cannot see would otherwise open a public
# tunnel to a server generating on CPU — minutes of audio at hours of wall clock, which is a worse
# outcome than not starting, and much harder to diagnose from the app.
_has_gpu = _torch_cuda
if _is_batch and not _has_gpu:
    print('=' * 70)
    print('  NO GPU ON THIS RUN — not serving.')
    print('  nvidia-smi: ' + ('exit {}'.format(_smi_rc) if _smi_rc is not None else 'did not run')
          + (' — {}'.format(_smi_note) if _smi_note else ''))
    print('  torch {}: cuda.is_available() = {}'.format(_torch_ver, _torch_cuda))
    print('  CUDA_VISIBLE_DEVICES = {!r}'.format(os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>')))
    if _smi_rc == 0:
        print('  GPU PRESENT BUT TORCH CANNOT USE IT — an install step in this notebook replaced')
        print("  Kaggle's CUDA build of torch with a CPU-only one. Fix that cell; the quota is")
        print('  not the problem here.')
    else:
        print('  KAGGLE GAVE THIS SESSION NO ACCELERATOR. The app always asks for one, so this is')
        print('  the scheduler declining: the weekly quota is spent, both GPU session slots are')
        print('  busy, or no GPU was free at that moment. That last case is common and transient —')
        print('  if the quota page still shows hours left, simply start again.')
        print('  Quota: https://www.kaggle.com/settings  (Accelerator usage).')
    print('  (A deliberate GPU-off push is just a cheap source update - nothing is wrong.)')
    print('=' * 70)
    print('Start the server from the app (Start server button) or run interactively with GPU on.')
else:
    args = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT), '--preview-method', 'auto']
    comfy = subprocess.Popen(args, cwd=COMFY, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)

    def _pump(p, tag):
        for line in p.stdout: print(f'[{tag}] {line}', end='')
    threading.Thread(target=_pump, args=(comfy,'comfy'), daemon=True).start()

    print('Waiting for ComfyUI to come up (first start loads nodes)...')
    for _ in range(90):
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=3); print('ComfyUI is up.'); break
        except Exception:
            time.sleep(3)
    else:
        print('WARNING: ComfyUI did not respond; check [comfy] logs above.')

    import urllib.request, urllib.error

    # ── Reliable public tunnel with self-healing ───────────────────────────────
    # The failure this fixes: cloudflared prints a *.trycloudflare.com URL and even registers an
    # edge connection, yet the Cloudflare edge never actually ROUTES the hostname, so the URL never
    # answers and the app times out. Quick tunnels are flaky per-process and QUIC (UDP) egress can
    # be throttled. So we: (1) probe our OWN public URL to confirm it truly routes, (2) auto-restart
    # cloudflared — first over QUIC, then over HTTP/2, which survives UDP throttling — and (3) fall
    # back to localhost.run (ssh) if cloudflared keeps failing. Only a URL that actually ANSWERS is
    # printed as ready, and every step is logged so a failure is diagnosable from the app's log tail.
    _url_re = re.compile(r'https://[-a-z0-9]+\.(?:trycloudflare\.com|lhr\.life|serveo\.net)')

    def _probe_public(url, timeout=8):
        # True iff the tunnel truly routes: ANY HTTP status < 500 back proves the edge reached our
        # server. A connection error/timeout, or Cloudflare's own 5xx (e.g. 530 = tunnel down),
        # means "not routed yet".
        try:
            with urllib.request.urlopen(url.rstrip('/') + '/', timeout=timeout) as r:
                return r.status < 500
        except urllib.error.HTTPError as he:
            return he.code < 500
        except Exception:
            return False

    def _pump(proc, holder, tag='tunnel'):
        def _run():
            for line in proc.stdout:
                print(f'[{tag}] {line}', end='')
                m = _url_re.search(line)
                if m and not holder.get('url'):
                    holder['url'] = m.group(0)
        threading.Thread(target=_run, daemon=True).start()

    def _spawn_cloudflared(protocol):
        args = ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}']
        if protocol:
            args += ['--protocol', protocol]
        print(f'[tunnel] launching cloudflared (protocol={protocol or "auto"})...', flush=True)
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _spawn_localhostrun():
        print('[tunnel] launching localhost.run over ssh...', flush=True)
        p = subprocess.Popen(
            ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'UserKnownHostsFile=/dev/null',
             '-o', 'ServerAliveInterval=30', '-R', f'80:localhost:{PORT}', 'nokey@localhost.run'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _bring_up(proc, holder, url_wait=30, route_wait=75):
        t0 = time.time()
        while time.time() - t0 < url_wait and not holder.get('url'):
            if proc.poll() is not None:
                print('[tunnel] process exited before printing a URL.', flush=True); return None
            time.sleep(1)
        url = holder.get('url')
        if not url:
            print(f'[tunnel] no URL within {url_wait}s.', flush=True); return None
        print(f'Tunnel URL: {url}', flush=True)
        print('Waiting for the edge to route it...', flush=True)
        t1 = time.time()
        while time.time() - t1 < route_wait:
            if proc.poll() is not None:
                print('[tunnel] tunnel process exited during the routing wait.', flush=True); return None
            if _probe_public(url):
                print(f'[tunnel] OK routed and answering after {int(time.time()-t1)}s.', flush=True)
                return url
            time.sleep(4)
        print(f'[tunnel] {url} never answered within {route_wait}s - treating as dead.', flush=True)
        return None

    _attempts = [('cf', 'quic'), ('cf', 'http2'), ('lhr', None)]
    active_proc = None; public_url = None
    for _i, (_kind, _proto) in enumerate(_attempts, 1):
        print(f'\n[tunnel] ===== attempt {_i}/{len(_attempts)}: {_kind} {_proto or ""} =====', flush=True)
        try:
            _p, _h = _spawn_cloudflared(_proto) if _kind == 'cf' else _spawn_localhostrun()
        except FileNotFoundError as _e:
            print(f'[tunnel] cannot launch ({_e}); skipping this attempt.', flush=True); continue
        _routed = _bring_up(_p, _h)
        if _routed:
            active_proc, public_url = _p, _routed; break
        try: _p.terminate()
        except Exception: pass
        time.sleep(2)

    print('\n' + '=' * 70)
    if public_url:
        print('  PASTE THIS INTO THE APP  ->  Settings -> ComfyUI server URL:')
        print(f'  {public_url}')
    else:
        print('  FAILED: no working public tunnel after all attempts. The local server is fine,')
        print('  but nothing outside can reach it - retry "Start & connect" from the app.')
    print('=' * 70, flush=True)

    if public_url and active_proc:
        # ── Idle-shutdown watchdog ──────────────────────────────────────
        # After IDLE_SHUTDOWN_MIN minutes with no ESTABLISHED connection to the server port, stop the
        # tunnel so a forgotten run stops burning GPU quota. App polling / liveness counts as activity.
        IDLE_SHUTDOWN_MIN = 15
        def _idle_watchdog():
            _port_hex = ':%04X' % PORT
            _last = time.time()
            while True:
                time.sleep(30)
                _active = False
                for _tbl in ('/proc/net/tcp', '/proc/net/tcp6'):
                    try:
                        with open(_tbl) as _f:
                            for _l in _f.readlines()[1:]:
                                _q = _l.split()
                                if _q[1].endswith(_port_hex) and _q[3] == '01':
                                    _active = True; break
                    except OSError:
                        _active = True
                    if _active: break
                if _active:
                    _last = time.time()
                elif time.time() - _last > IDLE_SHUTDOWN_MIN * 60:
                    print(f'[watchdog] No requests for {IDLE_SHUTDOWN_MIN} min - shutting down to save GPU quota.', flush=True)
                    try: active_proc.terminate()
                    except Exception: pass
                    return
        threading.Thread(target=_idle_watchdog, daemon=True).start()
        print(f'Keep this cell running. Idle watchdog armed: auto-stops after {IDLE_SHUTDOWN_MIN} min idle.', flush=True)
        active_proc.wait()